# Tạo FAISS index từ input.jsonl

Input cần đúng các field: `doc_id`, `chunk_order`, `char_start`, `char_end`, `chunk_text`.

- Train IVF bằng 200.000 record lấy mẫu ngẫu nhiên từ **toàn bộ file**, không phụ thuộc khoảng `doc_id`. File nhỏ hơn thì dùng tất cả.
- Chỉ thêm chunk trong khoảng `start_point <= doc_id <= end_point`. `end_point = -1` là không giới hạn trên.
- Output: một file `.faiss`. ID vector là số dòng JSONL trừ 1, giúp phân biệt các chunk cùng `doc_id`. Tra lại dòng `ID + 1` trong input để lấy nội dung; giữ nguyên thứ tự dòng của file này.


In [ ]:
!pip install torch
!pip install faiss-gpu
!pip install numpy

In [ ]:
import json
import faiss
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentenceTransformer("Alibaba-NLP/gte-multilingual-base", trust_remote_code=True, device=device)

quantizer = faiss.IndexFlatIP(768)
index = faiss.IndexIVFPQ(
    quantizer=quantizer, 
    d=768, 
    nlist=4096,
    M=64,
    nbits=8, 
    metric=faiss.METRIC_INNER_PRODUCT
)


In [ ]:
DATA_PATH = Path("train.txt")
EMBED_PATH = Path("train_embeddings.dat")

with DATA_PATH.open("r", encoding="utf-8") as f:
    nums = sum(1 for line in f if line.strip())

embeddings = np.memmap(
    EMBED_PATH,
    dtype=np.float32,
    mode="w+",
    shape=(nums, 768)
)


In [ ]:
def read_batches(): 
    batch = []
    with DATA_PATH.open("r", encoding="utf-8") as f:
        for line in f: 
            text = line.strip()
            if not text: 
                continue
            batch.append(text)
            if len(batch) == 128: 
                yield batch 
                batch = []

    if batch:
        yield batch

In [ ]:
offset = 0
with torch.inference_mode(): 
    for texts in read_batches(): 
        vectors = model.encode(
            texts,
            batch_size=128, 
            normalize_embeddings=True, 
            convert_to_numpy=True, 
            show_progress_bar=False
        )
        vectors = np.ascontiguousarray(
            vectors, 
            dtype=np.float32
        )
        n = len(vectors) 
        embeddings[offset:offset+n] = vectors
        offset+=n
        print(f"\rEmbedded: {offset:,}/{nums:,}", end="")   

embeddings.flush() 

train_vectors = np.memmap(
    EMBED_PATH, 
    dtype=np.float32, 
    mode="r", 
    shape=(nums, 768)
)

index.cp.seed = 42
index.train(train_vectors)
faiss.write_index(index, "index.faiss")